In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

ЗАГРУЗКА МОДЕЛИ GPT2

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "sshleifer/tiny-gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()
model.to("cpu")

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 2)
    (wpe): Embedding(1024, 2)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-1): 2 x GPT2Block(
        (ln_1): LayerNorm((2,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=6, nx=2)
          (c_proj): Conv1D(nf=2, nx=2)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((2,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=8, nx=2)
          (c_proj): Conv1D(nf=2, nx=8)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((2,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=2, out_features=50257, bias=False)
)

ПАРАМЕТРЫ МОДЕЛИ

In [ ]:
model.config

GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "dtype": "float32",
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 2,
  "n_head": 2,
  "n_inner": null,
  "n_layer": 2,
  "n_positions": 1024,
  "pad_token_id": null,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.18.0",
  "use_cache": true,
  "vocab_size": 50257
}

In [ ]:
inputs = tokenizer("Hello, my name is",return_tensors="pt")
inputs

{'input_ids': tensor([[15496,    11,   616,  1438,   318]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}

In [ ]:
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.1,
        top_p=0.8,
        pad_token_id=tokenizer.eos_token_id
    )
outputs

tensor([[15496,    11,   616,  1438,   318, 28625,  7806, 10323, 39261, 16046,
         21664, 23802, 29649, 37882, 50218,  9702, 28625, 21664, 17217,  8439,
         28625, 10323, 14903, 41950,  7806, 31351, 27965, 30873, 39261, 17217,
         45522, 31351, 34408, 45522, 10323, 40950,  7504, 14903, 16532,  8439,
         17192, 40950, 26964, 27965,  7504,   889, 14903, 29883, 34408, 29649,
         39261, 37882, 16046, 38361, 30873]])

In [ ]:
generated_text = tokenizer.decode(
    outputs[0].tolist(),
    skip_special_tokens=True
)
generated_text

'Hello, my name is heir Daniel Rh Hancock stairs TAmediately pawnJDRocket Brew heir TA reviewingpress heir Rh conservation Habit Daniel Motorola dispatch Prob Hancock reviewing circumcised Motorola Participation circumcised Rhoho Jr conservation credibilitypress vendorsohohibit dispatch Jrting conservation antibiotic Participation pawn HancockJD stairs intermittent Prob'

ЭКСПЕРИМЕНТ ТОКЕНИЗАЦИЯ

In [ ]:
english_input = "Machine learning is amazing!"
russian_input = "Машинное обучение — это интересно!"

russian_tokens = tokenizer(russian_input, return_tensors="pt")
english_tokens = tokenizer(english_input, return_tensors="pt")

russian_ids = russian_tokens["input_ids"][0].tolist()
english_ids = english_tokens["input_ids"][0].tolist()

russian_words = tokenizer.convert_ids_to_tokens(russian_ids)
english_words = tokenizer.convert_ids_to_tokens(english_ids)

df = pd.DataFrame({
    "Исходный текст": [russian_input, english_input],
    "Список ID": [russian_ids, english_ids],
    "Токены": [russian_words, english_words],
    "Кол-во токенов": [len(russian_ids), len(english_ids)]
})
df

,Исходный текст,Список ID,Токены,Кол-во токенов
0,Машинное обучение — это интересно!,"[140, 250, 16142, 141, 230, 18849, 22177, 2217...","[Ð, ľ, Ð°, Ñ, Ī, Ð¸, Ð½, Ð½, Ð¾, Ðµ, ĠÐ, ¾, Ð,...",38
1,Machine learning is amazing!,"[37573, 4673, 318, 4998, 0]","[Machine, Ġlearning, Ġis, Ġamazing, !]",5


In [ ]:
tokenizer.decode(english_ids)

'Machine learning is amazing!'

In [ ]:
experiments = {
    "1" : "Today the weather is",
    "2" : "Write exactly three colors separated by commas:",
    "3" : "He saw the bat and"
}

In [ ]:
def generate_text(prompt,mode="greedy",max_new_tokens=50,temperature=0.8,top_k=50,seed=42):
    inputs = tokenizer(prompt, return_tensors="pt")
    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": tokenizer.eos_token_id
    }

    if mode == "greedy":
        generation_kwargs["do_sample"] = False

    elif mode == "sampling":
        generation_kwargs["do_sample"] = True
        generation_kwargs["temperature"] = temperature
        generation_kwargs["top_k"] = top_k

        if seed is not None:
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)

    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **generation_kwargs
        )

    full_ids = outputs[0].tolist()
    input_length = inputs["input_ids"].shape[1]
    new_ids = full_ids[input_length:]
    full_text = tokenizer.decode(
        full_ids,
        skip_special_tokens=True
    )

    continuation = tokenizer.decode(
        new_ids,
        skip_special_tokens=True
    )
    return {
        "prompt": prompt,
        "mode": mode,
        "full_ids": full_ids,
        "new_ids": new_ids,
        "full_text": full_text,
        "continuation": continuation,
        "input_token_count": input_length,
        "generated_token_count": len(new_ids)
    }

In [ ]:
for key,prompt in experiments.items():
  greedy_res = generate_text(prompt)
  sampling_res = generate_text(prompt,"sampling")

  print(greedy_res["full_text"])
  print(sampling_res["full_text"])

Today the weather is factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors factors
Today the weather isozygGy incarcer448Mini praying Television Redux Singapore Boone omega Bend soy representations653 factors clearer brutality Pocket skillet Redux Bendacious courtyardMini Bend deflectived Singapore Tre courtyard TelevisionoblMostshowsived Medic skillet Television� Medic Singapore membership skillet courtyard factors clearerobl factors 236
Write exactly three colors separated by commas: stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs sta